<a href="https://colab.research.google.com/github/julianogueira3/Trabalho_mba_deeplearning/blob/main/trabalho_mba_deeplearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Introdução a modelos de deep learning Turma 9
#JÚLIA NOGUEIRA - 2427399
#JARLAN SILVA - 2427608
#ARTUR RODRIGUES - 2427753
#ISMAEL GÓES - 2428140
#Introdução: Sistema de Diagnóstico e Intervenção Educacional Baseado em IA
1. O que fizemos?
Desenvolvemos um ecossistema computacional integrado que une análise de dados (Pandas), Inteligência Artificial Generativa (LLM via Hugging Face) e automação de relatórios (ReportLab). O sistema processa indicadores de desempenho escolar (SAEB e taxas de alfabetização) e os cruza com informações qualitativas de infraestrutura para gerar planos de ação personalizados para escolas públicas.

2. Metodologia
O projeto foi estruturado em quatro pilares técnicos:

Tratamento de Dados: Utilização da biblioteca Pandas para filtrar e agregar microdados educacionais, permitindo identificar rapidamente as unidades escolares com maiores desafios.

Contextualização Híbrida: Criação de uma lógica de "Contexto Automático" que traduz números brutos em descritores textuais (ex: transformando uma nota baixa em "Desempenho muito baixo no SAEB").

Engenharia de Prompt e LLM: Integração com o modelo Qwen 2.5 7B através da API de Inferência do Hugging Face. Configuramos a IA para atuar como um "Especialista em Gestão Educacional", garantindo respostas técnicas e focadas em intervenção prática.

Persistência e Saída: Implementação de um módulo de histórico para rastreio de diagnósticos e geração automática de arquivos PDF para uso administrativo imediato.

3. Justificativa
Dados educacionais isolados em planilhas muitas vezes não se traduzem em melhorias reais por falta de braço técnico para analisá-los individualmente. Este sistema automatiza a ponte entre a evidência estatística e a tomada de decisão, permitindo que um gestor receba não apenas um gráfico, mas um cronograma de ações prioritárias.

4. Quais os objetivos?
Otimizar o tempo de diagnóstico das secretarias de educação.

Padronizar os critérios de intervenção pedagógica e de infraestrutura.

Prover soluções acionáveis, transformando problemas complexos (como baixa alfabetização ou teto avariado) em listas de verificação e cronogramas semanais.

5. O que alcançamos?
Conseguimos criar uma ferramenta funcional que entrega um Relatório de Diagnóstico Direto. O resultado final (como visto no log do sistema) apresenta:

Diagnóstico técnico preciso.

Mapeamento de problemas críticos.

Ações imediatas de alta prioridade.

Checklist de execução.

Cronograma estimado de implementação.

In [ ]:
!pip install requests python-dotenv transformers

In [ ]:
!pip install requests huggingface_hub python-dotenv

In [ ]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.5 MB/s eta 0:00:00


In [ ]:
import requests
from huggingface_hub import InferenceClient
from google.colab import userdata

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
TOKEN = userdata.get('HUGGING_FACE_HUB_TOKEN')

In [ ]:
import pandas as pd
df = pd.read_csv('/content/TS_ALUNO_2EF.csv', sep=';')

print("✅ CSV carregado")
print(df.shape)

✅ CSV carregado
(37104, 55)


In [ ]:
df_escolas = df.groupby("ID_ESCOLA").agg({
    "PROFICIENCIA_LP_SAEB": "mean",
    "IN_ALFABETIZADO": "mean"
}).reset_index()

display(df_escolas.head())

,ID_ESCOLA,PROFICIENCIA_LP_SAEB,IN_ALFABETIZADO
0,61398250,746.384800,0.363636
1,61398260,741.415053,0.476190
2,61398287,725.200576,0.432432
3,61398414,760.078372,0.500000
4,61398419,770.073043,0.760000


In [ ]:
def listar_escolas(df_escolas, limite=10):
    base = df_escolas.sort_values(by="PROFICIENCIA_LP_SAEB")
    print("🏫 Escolas mais críticas:\n")
    display(base.head(limite))

In [ ]:
def gerar_contexto_automatico(media, taxa):

    contexto = ""

    if media < 500:
        contexto += "Desempenho muito baixo no SAEB. "
    elif media < 600:
        contexto += "Desempenho abaixo da média. "
    else:
        contexto += "Desempenho adequado. "

    if taxa < 0.5:
        contexto += "Baixa taxa de alfabetização. "
    else:
        contexto += "Boa taxa de alfabetização. "

    return contexto

In [ ]:
def gerar_diagnostico_escola(nome, nota, infraestrutura, contexto):

    client = InferenceClient(
        model=MODEL_ID,
        token=TOKEN
    )

    prompt = f"""
    Você é um especialista em gestão educacional com foco em INTERVENÇÃO PRÁTICA.

    REGRAS:
    - NÃO seja genérico
    - Use exatamente os dados fornecidos
    - Transforme cada problema em ação direta

    Escola: {nome}
    Nota: {nota}
    Infraestrutura: {infraestrutura}
    Contexto: {contexto}

    FORMATO:

    1. Diagnóstico direto
    2. Problemas
    3. Ações imediatas (alta prioridade)
    4. Checklist
    5. Cronograma
    6. Score de risco (0 a 10)

    Se faltar algo (ex: cadeira), diga explicitamente para resolver.
    """

    response = client.chat_completion(
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=800,
        temperature=0.3
    )

    return response.choices[0].message.content

In [ ]:
def gerar_relatorio_hibrido(df_escolas, id_escola, infraestrutura_user, contexto_user):

    escola = df_escolas[df_escolas["ID_ESCOLA"] == id_escola]

    if escola.empty:
        return "❌ Escola não encontrada"

    media = escola["PROFICIENCIA_LP_SAEB"].values[0]
    taxa = escola["IN_ALFABETIZADO"].values[0]

    contexto_auto = gerar_contexto_automatico(media, taxa)

    contexto_final = f"""
    {contexto_auto}
    {contexto_user}
    """

    return gerar_diagnostico_escola(
        nome=f"Escola {id_escola}",
        nota=media / 100,
        infraestrutura=infraestrutura_user,
        contexto=contexto_final
    )

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from datetime import datetime

def salvar_pdf(relatorio, id_escola):

    nome = f"relatorio_{id_escola}_{datetime.now().strftime('%H%M%S')}.pdf"

    doc = SimpleDocTemplate(nome)
    styles = getSampleStyleSheet()

    conteudo = [Paragraph(linha, styles["Normal"]) for linha in relatorio.split("\n")]

    doc.build(conteudo)

    print(f"📄 PDF salvo: {nome}")

In [ ]:
historico = []

def salvar_historico(id_escola, relatorio):
    historico.append({
        "id": id_escola,
        "relatorio": relatorio,
        "data": datetime.now()
    })

def ver_historico():
    for h in historico:
        print(h["id"], h["data"])

In [ ]:
print("🎓 Sistema Final com IA + CSV + PDF + Histórico\n")

while True:

    listar_escolas(df_escolas)

    id_escola = input("\n🏫 ID da escola (ou sair): ")

    if id_escola == "sair":
        break

    if not id_escola.isdigit():
        print("❌ inválido")
        continue

    infraestrutura = input("🏗️ Infraestrutura: ")
    contexto = input("🌍 Contexto: ")

    print("\n⏳ Gerando...\n")

    relatorio = gerar_relatorio_hibrido(
        df_escolas,
        int(id_escola),
        infraestrutura,
        contexto
    )

    print(relatorio)

    salvar_pdf(relatorio, id_escola)
    salvar_historico(id_escola, relatorio)

    print("\n" + "="*60)

    opcao = input("hist / continuar: ")

    if opcao == "hist":
        ver_historico()

🎓 Sistema Final com IA + CSV + PDF + Histórico

🏫 Escolas mais críticas:



,ID_ESCOLA,PROFICIENCIA_LP_SAEB,IN_ALFABETIZADO
1157,61470633,631.707442,0.00
205,61411959,637.356434,0.00
353,61421750,646.890090,0.00
289,61418771,655.887879,0.00
229,61412989,663.533479,0.00
564,61436506,665.316942,0.00
114,61405341,665.385494,0.00
1175,61471355,665.624871,0.00
809,61451079,667.690530,0.00
1148,61470388,668.533307,0.05



⏳ Gerando...

1. **Diagnóstico direto**
   - A Escola 61470633 apresenta um desempenho adequado, mas com baixa taxa de alfabetização, indicando uma necessidade de melhoria na qualidade do ensino. A infraestrutura precária (teto caindo, ausência de janelas, mobiliário deficiente e inundações) afeta negativamente o ambiente de aprendizagem, o que pode contribuir para a baixa taxa de alfabetização.

2. **Problemas**
   - Teto caindo
   - Ausência de janelas (jmesa)
   - Mobiliário deficiente (somente cadeiras)
   - Inundações durante a chuva
   - Baixa taxa de alfabetização

3. **Ações imediatas (alta prioridade)**
   - **Segurança imediata**: Solicitar a interdição da escola até que as condições de segurança sejam corrigidas.
   - **Reparos urgentes**: Contratar uma empresa especializada para realizar os reparos necessários no teto e instalar janelas (jmesas).
   - **Mobiliário**: Adquirir mobiliário adequado (mesas e cadeiras de tamanho adequado para a idade dos alunos).
   - **Prevenç